In [12]:
import json
from pathlib import Path
from typing import Dict, Any, List

import sys, os
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils.json_utils import (
    normalize_chat_messages, anonymize_chat_messages,
    split_chat_into_batches, batch_to_dialog_text,
    build_chat_name_map
)


DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
SAMPLES_DIR = DATA_DIR / "samples"

PROC_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 20
MAX_DIALOG_CHARS = 20000
LANGUAGE = "ru"

In [13]:
def load_telegram_export(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

export_paths = sorted(RAW_DIR.glob("*.json"))
assert export_paths, f"No JSON exports found in {RAW_DIR}"

exports = [load_telegram_export(p) for p in export_paths]
len(exports), export_paths[:3]

(3,
 [PosixPath('../data/raw/result.json'),
  PosixPath('../data/raw/result2.json'),
  PosixPath('../data/raw/result3.json')])

In [14]:
def extract_chats(export_obj: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Returns a list of chat dicts with "messages".
    Handles common Telegram export structures.
    """

    chats_root = export_obj.get("chats")
    if isinstance(chats_root, dict):
        lst = chats_root.get("list")
        if isinstance(lst, list):
            return [c for c in lst if isinstance(c, dict) and "messages" in c]

    if "messages" in export_obj and isinstance(export_obj["messages"], list):
        return [export_obj]

    for key in ("chat", "channel", "group"):
        obj = export_obj.get(key)
        if isinstance(obj, dict) and "messages" in obj:
            return [obj]

    return []

all_chats = []
for e in exports:
    all_chats.extend(extract_chats(e))

len(all_chats), [c.get("name") for c in all_chats[:5]]

(3, ['Команда 3', 'CVWIDE', 'Talent Matcher // рабочий чат // Нить'])

In [15]:
chat_name_map = build_chat_name_map(all_chats)

def process_chat(chat):
    chat_norm = normalize_chat_messages(chat, drop_empty=True, drop_service=True, drop_emoji_only=True, dedupe_consecutive=True)
    chat_anon = anonymize_chat_messages(chat_norm, mask_urls=True, chat_name_map=chat_name_map)
    return chat_anon

processed_chats = [process_chat(c) for c in all_chats]
sum(len(c.get("messages", [])) for c in processed_chats)

3485

In [16]:
def chat_to_batches(chat: Dict[str, Any], batch_size: int = 20) -> List[Dict[str, Any]]:
    return split_chat_into_batches(chat, batch_size=batch_size)

all_batches = []
for c in processed_chats:
    all_batches.extend(chat_to_batches(c, batch_size=BATCH_SIZE))

len(all_batches), len(all_batches[0].get("messages", []))

(177, 20)

In [17]:
def make_dialogue_id(chat_meta: Dict[str, Any], idx: int) -> str:
    name = (chat_meta.get("name") or "chat").replace(" ", "_")
    return f"{name}__batch_{idx:05d}"

dataset = []
for chat in processed_chats:
    chat_meta = {k: v for k, v in chat.items() if k != "messages"}
    batches = split_chat_into_batches(chat, batch_size=BATCH_SIZE)

    for i, b in enumerate(batches):
        dialogue_id = make_dialogue_id(chat_meta, i)

        dialog_text = batch_to_dialog_text(b, max_chars=MAX_DIALOG_CHARS)

        item = {
            "dialogue_id": dialogue_id,
            "language": LANGUAGE,
            "dialogue": dialog_text,
            "messages": b.get("messages"),
            "meta": chat_meta,
        }
        dataset.append(item)

len(dataset), dataset[0]["dialogue_id"]

(177, 'Chat_001__batch_00000')

In [18]:
def count_speakers(messages: List[Dict[str, Any]]) -> int:
    s = set()
    for m in messages or []:
        fr = m.get("from")
        if isinstance(fr, str) and fr.strip():
            s.add(fr.strip())
    return len(s)

lengths = [len(d["dialogue"]) for d in dataset]
speakers = [count_speakers(d["messages"]) for d in dataset]

{
    "n_items": len(dataset),
    "avg_chars": sum(lengths) / max(1, len(lengths)),
    "min_chars": min(lengths) if lengths else 0,
    "max_chars": max(lengths) if lengths else 0,
    "avg_speakers": sum(speakers) / max(1, len(speakers)),
    "items_with_1_speaker": sum(1 for x in speakers if x <= 1),
}

{'n_items': 177,
 'avg_chars': 2085.3446327683614,
 'min_chars': 57,
 'max_chars': 12313,
 'avg_speakers': 4.2259887005649714,
 'items_with_1_speaker': 2}

In [19]:
def keep_item(item: Dict[str, Any], min_chars: int = 80, min_speakers: int = 2) -> bool:
    if len(item.get("dialogue", "")) < min_chars:
        return False
    if count_speakers(item.get("messages") or []) < min_speakers:
        return False
    return True

dataset_kept = [x for x in dataset if keep_item(x)]
len(dataset_kept), len(dataset_kept) / max(1, len(dataset))

(175, 0.9887005649717514)

In [20]:
def save_jsonl(items: List[Dict[str, Any]], path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")

out_path = PROC_DIR / "work_chats_dataset.jsonl"
save_jsonl(dataset, out_path)
out_path

PosixPath('../data/processed/work_chats_dataset.jsonl')

In [21]:
sample = dataset[:3]
sample_path = SAMPLES_DIR / "dataset_sample.jsonl"
save_jsonl(sample, sample_path)
sample_path

PosixPath('../data/samples/dataset_sample.jsonl')